<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/9_cnns_for_images_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Convolutional Neural Networks for Images · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. Your numbers will differ a little from run to run, so read the comments as "what
to look for", not as exact results.

This course is about text, but social scientists increasingly work with **images** too: protest
photos, politicians' Instagram posts, memes, news photography, satellite imagery. The workhorse
for images is the convolutional neural network (CNN). It's the same idea you met for text in
Bonus 7, but now the window slides across a two-dimensional grid of pixels instead of along a
sentence.

In this notebook you will:

- see that an image is just a grid of numbers, and what a convolution does to it
- train a CNN on clothing photos in **Keras** and in **PyTorch**
- look inside: which filters and features the network learned
- use a **pre-trained** model to label your own images and turn them into vectors (*image
  embeddings*), the visual counterpart of sentence embeddings
- think through what images-as-data can and can't do in social science, and the ethical issues

**Contents**

1. An image is a grid of numbers
2. Convolution by hand: filters that find edges
3. Pooling: zooming out
4. The data: Fashion-MNIST
5. A CNN in Keras
6. A CNN in PyTorch
7. Looking inside the network
8. Where does it go wrong?
9. Pre-trained models for your own images
10. Images as data in social science

> ⚠️ Turn on the GPU for Sections 5 to 9 (*Runtime → Change runtime type → T4 GPU*). On a CPU
> the notebook automatically trains on a smaller sample, so it still finishes in a few minutes.

## 0. Setup

Colab has everything we need preinstalled. As in the other bonus notebooks, Keras runs on the
**PyTorch backend**, so both stacks share one engine; the backend must be set before Keras is
imported. `torchvision` is PyTorch's companion library for images (datasets, pre-trained models,
image transformations), and `scikit-image` provides a few sample photos.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"     # must come before `import keras`

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torchvision
import keras

from scipy.signal import convolve2d
from skimage import data as sample_images
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
keras.utils.set_random_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Keras", keras.__version__, "on", keras.backend.backend(), "| PyTorch", torch.__version__,
      "| torchvision", torchvision.__version__, "| device:", device.upper())

## 1. An image is a grid of numbers

A grayscale image is a table of numbers, one per pixel, from 0 (black) to 255 (white). Here's a
classic test photo, and a zoom into a tiny 8×8 patch of it:

In [ ]:
camera = sample_images.camera()            # a 512 x 512 grayscale photo
print("Shape:", camera.shape, "| data type:", camera.dtype, "| values from", camera.min(), "to", camera.max())

patch = camera[100:108, 220:228]           # rows 100-107, columns 220-227
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
ax1.imshow(camera, cmap="gray")
ax1.add_patch(plt.Rectangle((220, 100), 8, 8, edgecolor="#34B233", facecolor="none", linewidth=2))
ax1.set_title("The whole image"); ax1.axis("off")
ax2.imshow(patch, cmap="gray", vmin=0, vmax=255)
for i in range(8):
    for j in range(8):
        ax2.text(j, i, patch[i, j], ha="center", va="center", fontsize=7,
                 color="white" if patch[i, j] < 128 else "black")
ax2.set_title("An 8 x 8 patch: just numbers"); ax2.axis("off")
plt.tight_layout()
plt.show()

A color image has three such grids stacked on top of each other, one each for **red**, **green**
and **blue**. These are called *channels*:

In [ ]:
astronaut = sample_images.astronaut()      # NASA portrait of astronaut Eileen Collins (public domain)
print("Shape:", astronaut.shape, "→ height x width x 3 color channels")

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
axes[0].imshow(astronaut); axes[0].set_title("color")
for ax, c, name in zip(axes[1:], range(3), ["red", "green", "blue"]):
    ax.imshow(astronaut[:, :, c], cmap="gray"); ax.set_title(f"{name} channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Convolution by hand: filters that find edges

A filter for images is a small grid of weights, typically 3×3. It slides over the image, and at
every position multiplies the pixels under it by its weights and adds them up, exactly like the
text filters in Bonus 7, but in two directions.

Two classic hand-designed filters detect **edges**: places where brightness changes sharply from
left to right, or from top to bottom.

In [ ]:
vertical_edges = np.array([[-1, 0, 1],
                           [-2, 0, 2],
                           [-1, 0, 1]])         # dark on the left, bright on the right
horizontal_edges = vertical_edges.T             # dark on top, bright below

img = camera.astype(float)
filtered = {
    "original": img,
    "vertical edges": np.abs(convolve2d(img, vertical_edges, mode="same")),
    "horizontal edges": np.abs(convolve2d(img, horizontal_edges, mode="same")),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (name, im) in zip(axes, filtered.items()):
    ax.imshow(im, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.tight_layout()
plt.show()

The vertical-edge filter lights up along the tripod legs, the tower and the outline of the coat;
the horizontal-edge filter along the horizon and the camera. A CNN's first layer learns dozens of filters like
these, **on its own**, from the training data. Deeper layers combine them: edges into corners and
textures, those into parts (a collar, a sleeve, an eye), those into objects.

> **✏️ Exercise 1**
>
> Design two filters of your own and apply them to `camera`:
>
> 1. a **blur**: a 5×5 filter in which every weight is 1/25 (each pixel becomes the average of
>    its neighborhood)
> 2. a **sharpen** filter: `[[0, -1, 0], [-1, 5, -1], [0, -1, 0]]`
>
> Show the original next to both results. Then explain in one sentence why a network might want
> a blur-like filter.

In [ ]:
# ✅ Solution
blur = np.ones((5, 5)) / 25
sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])

results_img = {"original": img,
               "blur": convolve2d(img, blur, mode="same"),
               "sharpen": np.clip(convolve2d(img, sharpen, mode="same"), 0, 255)}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (name, im) in zip(axes, results_img.items()):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()
# Blurring averages away pixel-level noise and fine texture, so later layers can respond to the
# larger shapes that matter for recognizing an object.

## 3. Pooling: zooming out

After a convolution, CNNs usually **pool**: they cut the image into small blocks (say 2×2) and
keep only the largest value in each block. The image gets four times smaller, keeps the strongest
responses, and becomes a little less sensitive to exactly *where* something is. It's the 2D
version of the max pooling you saw for text.

In [ ]:
def max_pool_2x2(im):
    h, w = im.shape[0] // 2 * 2, im.shape[1] // 2 * 2            # drop an odd last row/column
    return im[:h, :w].reshape(h // 2, 2, w // 2, 2).max(axis=(1, 3))

edges = filtered["vertical edges"]
pooled_once = max_pool_2x2(edges)
pooled_twice = max_pool_2x2(pooled_once)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im in zip(axes, [edges, pooled_once, pooled_twice]):
    ax.imshow(im, cmap="gray"); ax.set_title(f"{im.shape[0]} x {im.shape[1]}"); ax.axis("off")
plt.tight_layout()
plt.show()

## 4. The data: Fashion-MNIST

**Fashion-MNIST** is a standard practice dataset: 70,000 small (28×28) grayscale photos of
clothing in 10 categories, such as T-shirts, trousers and sneakers. It's small enough to train on
in minutes, and hard enough that the choice of model matters. We download it with `torchvision`
and use the same NumPy arrays for both stacks.

In [ ]:
train_raw = torchvision.datasets.FashionMNIST("data", train=True, download=True)
test_raw = torchvision.datasets.FashionMNIST("data", train=False, download=True)
class_names = train_raw.classes

# Pixel values from 0-255 to 0-1: networks train much better on small numbers
X_all = train_raw.data.numpy().astype("float32") / 255.0
y_all = train_raw.targets.numpy()
X_test = test_raw.data.numpy().astype("float32") / 255.0
y_test = test_raw.targets.numpy()

# Training and validation sets (the official test set stays untouched until the end).
# On a CPU we use a smaller training sample so that everything finishes in a few minutes.
N_TRAIN = 50_000 if device == "cuda" else 20_000
X_train, y_train = X_all[:N_TRAIN], y_all[:N_TRAIN]
X_val, y_val = X_all[50_000:], y_all[50_000:]                 # the last 10,000 training images

print(f"train {X_train.shape}, validation {X_val.shape}, test {X_test.shape}")
print("Classes:", class_names)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(13, 3.8))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(X_train[i], cmap="gray_r")
    ax.set_title(class_names[y_train[i]], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

### A baseline without convolutions

As always, a simple baseline first: logistic regression on the 784 raw pixel values, each pixel
treated as a separate feature. It's the image equivalent of a bag of words: it has no idea which
pixels are neighbors.

In [ ]:
start = time.time()
pixel_lr = LogisticRegression(max_iter=500)
pixel_lr.fit(X_train.reshape(len(X_train), -1)[:10_000], y_train[:10_000])   # 10,000 images is plenty here
val_acc_lr = accuracy_score(y_val, pixel_lr.predict(X_val.reshape(len(X_val), -1)))
print(f"Logistic regression on pixels: validation accuracy {val_acc_lr:.3f} ({time.time() - start:.0f} s)")
results = {"Logistic regression (pixels)": val_acc_lr}

## 5. A CNN in Keras

A small but typical CNN: two rounds of *convolution → pooling*, then a dense output layer.

| Layer | What it does | Output shape |
|---|---|---|
| `Conv2D(32, 3)` | 32 filters of 3×3, ReLU | 26 × 26 × 32 |
| `MaxPooling2D(2)` | keep the max of every 2×2 block | 13 × 13 × 32 |
| `Conv2D(64, 3)` | 64 filters, each looking at all 32 feature maps | 11 × 11 × 64 |
| `MaxPooling2D(2)` | | 5 × 5 × 64 |
| `Flatten` + `Dropout` + `Dense(10)` | one probability per class | 10 |

Keras expects images as *height × width × channels*, so we add a channel axis of size 1.

In [ ]:
Xk_train, Xk_val, Xk_test = X_train[..., None], X_val[..., None], X_test[..., None]   # add channel axis

keras.utils.set_random_seed(SEED)
keras_cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(32, kernel_size=3, activation="relu"),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Conv2D(64, kernel_size=3, activation="relu"),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation="softmax"),
])
keras_cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
keras_cnn.summary()

In [ ]:
start = time.time()
keras_history = keras_cnn.fit(Xk_train, y_train, validation_data=(Xk_val, y_val),
                              epochs=5, batch_size=64, verbose=2)
print(f"Training took {time.time() - start:.0f} s")
results["Keras CNN"] = keras_history.history["val_accuracy"][-1]

Compare the validation accuracy with the pixel baseline. Even this small CNN, trained for just
five epochs, beats it by several points. With more filters, batch normalization (Exercise 3) or
longer training, CNNs reach 90% and more on this dataset, a level logistic regression on raw
pixels doesn't get near. Unlike with short bill titles, where a few keywords did most of the work,
here the architecture matters: a CNN knows that neighboring pixels belong together, and that a
pattern means the same thing wherever it appears in the image.

> **✏️ Exercise 2**
>
> Check the output shapes in the table above by hand, and against `keras_cnn.summary()`:
>
> 1. Why does a 3×3 convolution turn 28×28 into 26×26? (Think of how many positions a 3-pixel
>    window has along a 28-pixel row.)
> 2. How many weights does the second `Conv2D` layer have? (Each of its 64 filters is 3×3 and
>    looks at all 32 feature maps from the layer before, plus one bias per filter.)

In [ ]:
# ✅ Solution
# 1. A 3-wide window fits 28 - 3 + 1 = 26 times along each side (no padding).
#    Pooling halves it (rounding down): 26 -> 13; then 13 - 3 + 1 = 11 -> pooled to 5.
# 2. 64 filters x (3 x 3 x 32 weights) + 64 biases:
print(64 * (3 * 3 * 32) + 64)          # 18,496, matching the summary
for layer in keras_cnn.layers:
    print(f"{layer.name:<16} {str(layer.output.shape):<22} {layer.count_params():>8,} parameters")

## 6. A CNN in PyTorch

The same architecture in PyTorch. One difference to watch: PyTorch puts the channel axis
**first** (*channels × height × width*), so the images get shape `[batch, 1, 28, 28]`.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),    # -> 32 x 13 x 13
            nn.Conv2d(32, 64, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),   # -> 64 x 5 x 5
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(64 * 5 * 5, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def make_loader(X, y, shuffle, batch_size=64):
    ds = TensorDataset(torch.tensor(X[:, None]), torch.tensor(y, dtype=torch.long))   # add channel axis
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader = make_loader(X_val, y_val, shuffle=False)
test_loader = make_loader(X_test, y_test, shuffle=False)

torch.manual_seed(SEED)
torch_cnn = FashionCNN().to(device)
print("Parameters:", sum(p.numel() for p in torch_cnn.parameters()))

In [ ]:
def predict_torch(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            preds.append(model(xb.to(device)).argmax(dim=1).cpu())
    return torch.cat(preds).numpy()


def train_torch(model, epochs=5, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running += loss.item() * len(yb)
        val_acc = accuracy_score(y_val, predict_torch(model, val_loader))
        print(f"epoch {epoch + 1}: training loss {running / len(train_loader.dataset):.4f}, "
              f"validation accuracy {val_acc:.3f}")
    return val_acc


start = time.time()
results["PyTorch CNN"] = train_torch(torch_cnn)
print(f"Training took {time.time() - start:.0f} s")

> **✏️ Exercise 3**
>
> Two common ways to improve a CNN. Try each in PyTorch (copy `FashionCNN` into a new class) and
> compare the validation accuracy after 5 epochs:
>
> 1. **More filters**: 64 and 128 instead of 32 and 64.
> 2. **Batch normalization**: add `nn.BatchNorm2d(32)` right after the first `Conv2d` and
>    `nn.BatchNorm2d(64)` after the second. Batch normalization rescales each layer's outputs
>    during training, which often makes training faster and more stable.

In [ ]:
# ✅ Solution
class WiderCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(128 * 5 * 5, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

class BatchNormCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.5), nn.Linear(64 * 5 * 5, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

for name, cls in [("More filters", WiderCNN), ("Batch normalization", BatchNormCNN)]:
    torch.manual_seed(SEED)
    print(name)
    results[f"PyTorch CNN, {name.lower()}"] = train_torch(cls().to(device))
# Both typically add a little accuracy. Batch norm mainly speeds up learning in the first epochs.

## 7. Looking inside the network

### The learned filters

Here are the 32 filters of the first layer of the PyTorch model, each a 3×3 grid of weights
(light = positive, dark = negative). Many look like little edge or gradient detectors, the kind of
thing we designed by hand in Section 2, but learned from the data.

In [ ]:
filters = torch_cnn.features[0].weight.detach().cpu().numpy()[:, 0]     # [32, 3, 3]
fig, axes = plt.subplots(4, 8, figsize=(9, 4.8))
for ax, f in zip(axes.ravel(), filters):
    ax.imshow(f, cmap="gray"); ax.axis("off")
plt.suptitle("First-layer filters (3 x 3)")
plt.tight_layout()
plt.show()

### Feature maps

More revealing is what the filters *do* to an image. Each filter produces a **feature map**:
where in the image its pattern was found.

In [ ]:
def show_feature_maps(model, image, n=16):
    model.eval()
    with torch.no_grad():
        maps = torch.relu(model.features[0](torch.tensor(image[None, None]).to(device)))[0].cpu().numpy()
    fig, axes = plt.subplots(2, n // 2 + 1, figsize=(13, 3.6))
    axes[0, 0].imshow(image, cmap="gray_r"); axes[0, 0].set_title("input", fontsize=8)
    axes[1, 0].axis("off")
    for k, ax in enumerate(axes[:, 1:].T.ravel()[:n]):
        ax.imshow(maps[k], cmap="viridis"); ax.set_title(f"filter {k}", fontsize=7)
    for ax in axes.ravel():
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

show_feature_maps(torch_cnn, X_val[0])

> **✏️ Exercise 4**
>
> Show the feature maps for a **sneaker** and for a **shirt** from the validation set (find one of
> each with `np.where(y_val == class_names.index("Sneaker"))[0][0]`). Do some filters respond
> mostly to outlines, others to textures or to particular parts?

In [ ]:
# ✅ Solution
for cls in ["Sneaker", "Shirt"]:
    idx = np.where(y_val == class_names.index(cls))[0][0]
    print(cls)
    show_feature_maps(torch_cnn, X_val[idx])
# Typically some maps highlight left or right outlines, some top or bottom edges, and some
# respond to the whole bright area of the garment. Second-layer maps (not shown) combine these
# into more specific parts.

## 8. Where does it go wrong?

Accuracy hides *which* mistakes a model makes. The confusion matrix shows them, for the Keras
model on the validation set:

In [ ]:
keras_val_pred = keras_cnn.predict(Xk_val, verbose=0).argmax(axis=1)

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(y_val, keras_val_pred), display_labels=class_names).plot(
    ax=ax, cmap="Greens", colorbar=False, xticks_rotation=45)
ax.set_title("Keras CNN, validation set")
plt.tight_layout()
plt.show()

The typical confusions are among the tops (*T-shirt/top*, *Shirt*, *Pullover*, *Coat*), which
look alike at 28×28 pixels. Would a person do much better on these images? Sometimes the "error"
says more about the category scheme than about the model, the same lesson as the policy areas
on Day 3.

> **✏️ Exercise 5**
>
> 1. Find the three most frequent confusions (pairs of *true class → predicted class*, excluding the
>    diagonal) in the confusion matrix.
> 2. Show eight misclassified validation images with their true and predicted labels. Would you
>    have gotten them right?

In [ ]:
# ✅ Solution
cm = confusion_matrix(y_val, keras_val_pred)
np.fill_diagonal(cm, 0)
pairs = sorted(((cm[i, j], class_names[i], class_names[j]) for i in range(10) for j in range(10)), reverse=True)
for n, true_c, pred_c in pairs[:3]:
    print(f"{n:>4} x  {true_c} → predicted as {pred_c}")

wrong = np.where(keras_val_pred != y_val)[0][:8]
fig, axes = plt.subplots(1, 8, figsize=(14, 2.6))
for ax, i in zip(axes, wrong):
    ax.imshow(X_val[i], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"true: {class_names[y_val[i]]}\npred: {class_names[keras_val_pred[i]]}", fontsize=7)
plt.tight_layout(); plt.show()
# Many mistakes are genuinely ambiguous at this resolution: is it a shirt or a T-shirt?

### The test set, once

In [ ]:
test_scores = {
    "Logistic regression (pixels)": accuracy_score(y_test, pixel_lr.predict(X_test.reshape(len(X_test), -1))),
    "Keras CNN": accuracy_score(y_test, keras_cnn.predict(Xk_test, verbose=0).argmax(axis=1)),
    "PyTorch CNN": accuracy_score(y_test, predict_torch(torch_cnn, test_loader)),
}
pd.DataFrame({"validation accuracy": pd.Series(results), "test accuracy": pd.Series(test_scores)}).round(3)

## 9. Pre-trained models for your own images

You'll rarely train an image model from scratch for research. Just as with BERT for text, you
start from a model **pre-trained** on a huge image collection. The classic one is **ImageNet**:
over a million photos in 1,000 categories, from *tabby cat* to *space shuttle*. Two ways to use
such a model, which mirror the usage modes from Day 3:

1. **Off the shelf**: let it label images with its 1,000 ImageNet categories.
2. **As a feature extractor**: cut off the last layer and use the layer before it as a vector
   for each image, an **image embedding**. Similar images get similar vectors, so you can cluster
   them, search them, or train a simple classifier on top with your own categories.

### Off the shelf, in PyTorch: ResNet-18

`torchvision` comes with many pre-trained models. Each one has a set of *weights* that also
knows the right image preprocessing and the category names.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

weights = ResNet18_Weights.DEFAULT                 # ImageNet-trained weights (downloads ~45 MB)
resnet = resnet18(weights=weights).to(device).eval()
preprocess = weights.transforms()                  # resize, crop to 224 x 224, normalize colors
categories = weights.meta["categories"]            # the 1,000 ImageNet class names

photos = {
    "astronaut": sample_images.astronaut(),
    "coffee": sample_images.coffee(),
    "cat": sample_images.chelsea(),
    "rocket": sample_images.rocket(),
}

def to_tensor(rgb_array):
    # NumPy image (height x width x 3, values 0-255) -> preprocessed tensor for the model
    return preprocess(torch.tensor(rgb_array).permute(2, 0, 1))

batch = torch.stack([to_tensor(p) for p in photos.values()]).to(device)
with torch.no_grad():
    probs = torch.softmax(resnet(batch), dim=1).cpu()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8))
for ax, (name, photo), p in zip(axes, photos.items(), probs):
    top = p.topk(3)
    ax.imshow(photo); ax.axis("off")
    ax.set_title("\n".join(f"{categories[i]} ({v:.2f})" for v, i in zip(top.values, top.indices)), fontsize=8)
plt.tight_layout()
plt.show()

Notice that the model can only answer in terms of its 1,000 ImageNet categories. It has a
*space shuttle* category but nothing for "astronaut" or "politician", and its categories were
chosen by computer scientists, not with social science concepts in mind. Off-the-shelf labels are
rarely the measure you want; embeddings usually are more useful.

### As a feature extractor: image embeddings

Replace the final classification layer with an identity function, and ResNet-18 returns a
512-number vector for every image.

In [ ]:
resnet.fc = nn.Identity()                           # drop the 1,000-class output layer
with torch.no_grad():
    image_vectors = resnet(batch).cpu().numpy()     # one 512-dimensional vector per photo
print("Image embeddings:", image_vectors.shape)

norm = image_vectors / np.linalg.norm(image_vectors, axis=1, keepdims=True)
pd.DataFrame(norm @ norm.T, index=photos.keys(), columns=photos.keys()).round(2)

That's a cosine similarity matrix, just like the ones for word and sentence embeddings. With a
few hundred of your own images, you could now cluster them (as in Art Dewulf's session, but for
pictures) or train a logistic regression on the vectors to classify them into *your* categories,
with far fewer labeled examples than training from scratch would need.

### The same in Keras

`keras.applications` has its own collection of pre-trained models. Here's MobileNetV2, a small
model designed for phones, used as a feature extractor (`include_top=False` drops the output
layer; `pooling="avg"` turns the final feature maps into one vector per image).

In [ ]:
mobilenet = keras.applications.MobileNetV2(weights="imagenet", include_top=False,
                                           pooling="avg", input_shape=(224, 224, 3))

def keras_prepare(rgb_array):
    resized = keras.ops.image.resize(rgb_array.astype("float32"), (224, 224))   # -> 224 x 224 x 3
    return keras.applications.mobilenet_v2.preprocess_input(keras.ops.convert_to_numpy(resized))

keras_batch = np.stack([keras_prepare(p) for p in photos.values()])
keras_vectors = mobilenet.predict(keras_batch, verbose=0)
print("MobileNetV2 image embeddings:", keras_vectors.shape)

> **✏️ Exercise 6**
>
> 1. Compute the cosine similarity matrix for the MobileNetV2 vectors. Do the two models agree on
>    which photos are most alike?
> 2. Add two more sample photos to the comparison, for example `sample_images.coins()` (grayscale,
>    so first stack it into three channels with `np.stack([im] * 3, axis=-1)`) and
>    `sample_images.immunohistochemistry()`. Which existing photo is each one closest to?

In [ ]:
# ✅ Solution
# 1. MobileNetV2 similarities
kn = keras_vectors / np.linalg.norm(keras_vectors, axis=1, keepdims=True)
display(pd.DataFrame(kn @ kn.T, index=photos.keys(), columns=photos.keys()).round(2))

# 2. Two extra photos, compared with ResNet-18 vectors
extra = {"coins": np.stack([sample_images.coins()] * 3, axis=-1),
         "microscope": sample_images.immunohistochemistry()}
all_photos = {**photos, **extra}
with torch.no_grad():
    v = resnet(torch.stack([to_tensor(p) for p in all_photos.values()]).to(device)).cpu().numpy()
v = v / np.linalg.norm(v, axis=1, keepdims=True)
sims = pd.DataFrame(v @ v.T, index=all_photos.keys(), columns=all_photos.keys())
for name in extra:
    print(f"{name} is most similar to: {sims[name].drop(name).idxmax()}")
# Exact numbers differ between the two models, but the broad pattern of which photos are alike
# should be similar. Image embeddings capture visual similarity (colors, textures, shapes),
# which is not always the kind of similarity your research question is about.

## 10. Images as data in social science

Some examples of what researchers do with these tools:

- **Protest and collective action**: detecting protest events in social media images, and
  measuring features such as crowd size or violence (Zhang & Pan, 2019; Joo & Steinert-Threlkeld,
  2022).
- **Political self-presentation**: how politicians portray themselves on Instagram and in
  campaign material, for example with family, in uniform, or among crowds.
- **Media content**: who appears in news photos, and how often.
- **Economic development**: estimating poverty from satellite images of roads, roofs and night
  lights (Jean et al., 2016).

Newer **vision-language models** such as CLIP (Radford et al., 2021) go a step further: like the
zero-shot text classifier on Day 3, they can sort images into categories you describe in words,
without training, and they power the multimodal LLMs you'll meet on Day 4. The same validation
discipline applies.

### Ethics: be careful with images of people

- **Faces are personal data.** Under the GDPR, photos of identifiable people are personal data,
  and face recognition counts as biometric data, which gets extra protection. Talk to your data
  steward before you start.
- **Models are biased.** Commercial face analysis systems have been shown to make far more errors
  for some groups than others (Buolamwini & Gebru, 2018). Any measure built on such a model can
  carry that bias into your results.
- **Context and consent.** A public Instagram post was not shared so that it could be analyzed in a
  research project. Platform terms of service and research ethics boards both have views on that.
- **Validate, as always.** Hand-code a sample, check agreement, and look at the errors, exactly as
  with text.

### Further reading

- Williams, N. W., Casas, A., & Wilkerson, J. D. (2020). *Images as data for social science
  research: An introduction to convolutional neural nets for image classification*. Cambridge
  University Press (Elements in Quantitative and Computational Methods for the Social Sciences).
- Joo, J., & Steinert-Threlkeld, Z. C. (2022). Image as data: Automated content analysis for visual
  presentations of political actors and events. *Computational Communication Research, 4*(1).
- Zhang, H., & Pan, J. (2019). CASM: A deep-learning approach for identifying collective action
  events with text and image data from social media. *Sociological Methodology, 49*(1).
- Jean, N., Burke, M., Xie, M., Davis, W. M., Lobell, D. B., & Ermon, S. (2016). Combining satellite
  imagery and machine learning to predict poverty. *Science, 353*(6301).
- Buolamwini, J., & Gebru, T. (2018). Gender shades: Intersectional accuracy disparities in
  commercial gender classification. *Proceedings of the Conference on Fairness, Accountability and
  Transparency (FAT\*) 2018*.
- Radford, A., et al. (2021). Learning transferable visual models from natural language
  supervision. *Proceedings of ICML 2021*. (CLIP.)